List All Imports

In [1]:
import os
from chess import pgn
from tqdm import tqdm
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import Board
import tensorflow as tf
import pickle
import json
import pandas as pd

Load raw games and preprocess them for data set

In [2]:
# get the game files
# files = [file for file in os.listdir("simulated_games_filtered_PGN") if file.endswith(".pgn")]
df = pd.read_csv('lichess_training_data.csv')
print(f"Loaded {len(df)} games from Lichess")
print(df.head())

Loaded 415 games from Lichess
          White        Black  WhiteElo  BlackElo Result        Date  \
0  brzigonzalez     worms123      2174      2013    0-1  ????.??.??   
1        nesa10      futurum      2054      2136    0-1  ????.??.??   
2        nesa10    olman2011      2046      2041    0-1  ????.??.??   
3        nesa10        alpay      2036      2057    0-1  ????.??.??   
4       vit2014  ali13agheri      2148      2056    0-1  ????.??.??   

                  Event  ECO  \
0  Rated Classical game  C40   
1  Rated Classical game  C70   
2  Rated Classical game  C00   
3  Rated Classical game  B50   
4  Rated Classical game  A29   

                                               Moves  
0  1. e4 e5 2. Nf3 f5 3. Qe2 fxe4 4. Nxe5 Nf6 5. ...  
1  1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. Bc4 Nf6 5. d3...  
2  1. e4 e6 2. Nf3 d5 3. exd5 exd5 4. d4 Nf6 5. B...  
3  1. e4 c5 2. Nf3 d6 3. Nc3 Nc6 4. Bb5 Bd7 5. Bx...  
4  1. c4 Nc6 2. Nc3 e5 3. Nf3 Nf6 4. g3 Bc5 5. Bg...  


In [3]:
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

def create_input_from_lichess_csv(df):
    """Convert Lichess CSV to training data"""
    X = []
    y = []
    skipped = 0
    
    print("Converting Lichess games to board positions...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            moves_str = str(row['Moves'])
            tokens = moves_str.strip().split()
            moves_san = [tok for tok in tokens if not tok.endswith('.')]
            
            board = Board()
            
            for san in moves_san:
                move = board.parse_san(san)
                move_uci = move.uci()
                
                X.append(board_to_matrix(board))
                y.append(move_uci)
                board.push(move)
        
        except Exception as e:
            skipped += 1
            continue
    
    print(f"\nCreated {len(X)} training samples (skipped {skipped} games)")
    return X, y

# Run it
X_raw, y_raw = create_input_from_lichess_csv(df)

Converting Lichess games to board positions...


100%|██████████| 415/415 [00:00<00:00, 505.33it/s]


Created 30743 training samples (skipped 0 games)


In [4]:
# Load your pre-trained model's move encoding
with open('simulated_filtered_model/move_to_int.pkl', 'rb') as f:
    move_to_int = pickle.load(f)

print(f"Loaded move encoding with {len(move_to_int)} unique moves")

# Encode the moves, only keep those in the existing encoding
y_encoded = []
X_filtered = []
skipped_moves = 0

for move_uci, board_matrix in zip(y_raw, X_raw):
    if move_uci in move_to_int:
        y_encoded.append(move_to_int[move_uci])
        X_filtered.append(board_matrix)
    else:
        skipped_moves += 1

# 200k sub-samples limit
import random
random.seed(42)
indices = random.sample(range(len(X_filtered)), min(200000, len(X_filtered)))

X_filtered = [X_filtered[i] for i in indices]
y_encoded = [y_encoded[i] for i in indices]

X = np.array(X_filtered)
y = np.array(y_encoded)

print(f"\nFinal dataset:")
print(f"  X shape: {X.shape}")
print(f"  y shape: {y.shape}")
print(f"  Skipped unknown moves: {skipped_moves}")

# Convert to one-hot
y = tf.keras.utils.to_categorical(y, num_classes=len(move_to_int))
print(f"  y one-hot shape: {y.shape}")

Loaded move encoding with 1945 unique moves

Final dataset:
  X shape: (30743, 8, 8, 12)
  y shape: (30743,)
  Skipped unknown moves: 0
  y one-hot shape: (30743, 1945)


In [5]:
pretrained_model = tf.keras.models.load_model("simulated_filtered_model/SSMF_50EPOCHS.keras")

print("Layers in pretrained model:")
for i, layer in enumerate(pretrained_model.layers):
    print(f"{i}: {layer.name}")

conv_base = tf.keras.Sequential(pretrained_model.layers[:1])

conv_base.build((None, 8, 8, 12)) 

conv_base.trainable = False

new_model = tf.keras.Sequential([
        conv_base,
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(2048, activation='relu', name='new_dense1'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1024, activation='relu', name='new_dense2'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(len(move_to_int), activation='softmax', name='new_output')
    ])

Layers in pretrained model:
0: conv2d_2
1: conv2d_3
2: flatten_1
3: dense_2
4: dense_3


In [6]:
new_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

new_model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

print("Training on Lichess data...")
new_model.fit(X, y, epochs=50, validation_split=0.1, batch_size=64, callbacks=callbacks)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense1 (Dense)              │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense2 (Dense)              │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_output (Dense)              │ (None, 1945)           │     1,993,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,368,985 (31.93 MB)

 Trainable params: 8,362,009 (31.90 MB)

 Non-trainable params: 6,976 (27.25 KB)

Training on Lichess data...
Epoch 1/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.0153 - loss: 6.6400 - val_accuracy: 0.0189 - val_loss: 6.2904
Epoch 2/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.0272 - loss: 6.2655 - val_accuracy: 0.0377 - val_loss: 6.1923
Epoch 3/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 18s 41ms/step - accuracy: 0.0366 - loss: 6.1512 - val_accuracy: 0.0498 - val_loss: 6.1109
Epoch 4/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - accuracy: 0.0509 - loss: 6.0341 - val_accuracy: 0.0602 - val_loss: 6.0156
Epoch 5/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - accuracy: 0.0614 - loss: 5.9139 - val_accuracy: 0.0751 - val_loss: 5.9233
Epoch 6/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 21s 48ms/step - accuracy: 0.0728 - loss: 5.7908 - val_accuracy: 0.0774 - val_loss: 5.8455
Epoch 7/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 22s 51ms/step - accuracy: 0.0848 - loss: 5.6647 - val_accuracy: 0.0852 - val_loss: 5.7622
Epoch 8/50
433/433 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - accura

Compile and train new layers in model

In [7]:
# Create directory
os.makedirs('simulated_filtered_model_lichess_transfer', exist_ok=True)

# save the model
new_model.save("simulated_filtered_model_lichess_transfer/SSMF_50EPOCHS.keras")

# save the encoding
with open("simulated_filtered_model_lichess_transfer/move_to_int.pkl", "wb") as f:
    pickle.dump(move_to_int, f)

int_to_move = {v: k for k, v in move_to_int.items()}
with open("simulated_filtered_model_lichess_transfer/int_to_move.pkl", "wb") as f:
    pickle.dump(int_to_move, f)

# configuration 
config = {
    "data_source": "Lichess July 2014 (1800+ Elo, 15k games)",
    "pre_trained_model": "simulated_filtered_model/SSMF_50EPOCHS.keras",
    "epochs": 50,
    "batch_size": 64,
    "validation_split": 0.1,
    "optimizer": "Adam",
    "learning_rate": 0.0001,
    "input_shape": (8, 8, 12),
    "transfer_learning": True,
    "frozen_layers": "conv2d, conv2d_1, flatten",
    "early_stopping": True,
    "early_stopping_patience": 10
}

# save the configurations
with open("simulated_filtered_model_lichess_transfer/train_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("✓ Model saved to simulated_filtered_model_lichess_transfer/")

✓ Model saved to simulated_filtered_model_lichess_transfer/


Save model and related data to training and configuration